# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ayustar/flyrank-ml-internship-starter/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import os, json
import pandas as pd

if not os.path.exists("docs/results.json"):
    !git clone https://github.com/Ayustar/flyrank-ml-internship-starter.git
    %cd flyrank-ml-internship-starter

R = json.load(open("docs/results.json"))
print("loaded results for", R["n_pages"], "pages")

Cloning into 'flyrank-ml-internship-starter'...
remote: Enumerating objects: 195, done.
remote: Counting objects: 100% (195/195), done.
remote: Compressing objects: 100% (151/151), done.
remote: Total 195 (delta 87), reused 92 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (195/195), 1.92 MiB | 7.93 MiB/s, done.
Resolving deltas: 100% (87/87), done.
/content/flyrank-ml-internship-starter
loaded results for 30000 pages


## 1. Question

*The research question and the decision it supports.*

Question: Can a model rank pages with falling impressions better than a simple hand-written rule when it is tested on clients it has never seen? “Better” means a higher rank-quality score (AUC, precision@50) on the same pages. The output is a ranked review queue for human editors, not an automatic edit list.
Paper: section 1 (Introduction) and the Abstract. Notebooks: w04_baseline_score.ipynb frames the baseline rule.

## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

Data: the anonymised FlyRank starter dataset, filtered to pages with impressions and at least 90 days of age, one row per page. No client names, domains, URLs, queries, or product flags. The label is a current-window proxy (trend_direction == "down") used only for evaluation.
Paper: section 2. Notebooks: w04, w05_model.ipynb.

In [2]:
print("pages:", R["n_pages"], "| base decline rate:", R["base_rate"])

pages: 30000 | base decline rate: 0.542


## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

Method: a baseline rule (stale 90+ days and 500+ impressions) against a Random Forest, scored with 5-fold cross-validation grouped by client. Leak audit: single-column AUC screening and a feature-importance review. The 30-day impression, click, and session columns were removed because the label is built from those windows.
Paper: section 3. Notebooks: w05_model.ipynb, w06_validation_audit.ipynb, w07_action_playbook.ipynb.

In [3]:
print("features used:", R["n_features"])
print("removed as label-window leaks:", R["leaks"])

features used: 23
removed as label-window leaks: []


## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

Method	AUC	Average precision	Precision@50
Baseline rule	0.530	n/a	0.580
Random Forest (client-grouped)	0.687	0.687	0.580
Random picking	0.500	n/a	0.542
Under a random split, the same model scored AUC 0.778 and precision@50 of 1.000. The gap to the client-grouped numbers shows why a grouped design matters.

In [4]:
res = pd.DataFrame({
    "method": ["Baseline rule", "Random Forest (client-grouped)", "Random picking"],
    "AUC": [R["base_auc"], R["model_auc"], 0.5],
    "Precision@50": [R["base_p50"], R["model_p50"], R["base_rate"]],
}).set_index("method")
print(res)
print("\nsame model, random split: AUC", R["random_cv_auc"], "| P@50", R["random_cv_p50"])

                                  AUC  Precision@50
method                                             
Baseline rule                   0.530         0.580
Random Forest (client-grouped)  0.687         0.580
Random picking                  0.500         0.542

same model, random split: AUC 0.778 | P@50 1.0


## 5. Limitations

*What this work cannot claim.*

The label is a current-window proxy, not a future outcome.
The data is a 30,000-page starter slice, so results are directional.
The 90-day totals still overlap the label window.
One validation design (client-grouped); a time-aware test would be stronger.
Observational only: nothing here shows a refresh causes recovery.

## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

Playbook: one reason code and one action per page, ranked by model risk (60%) and visibility (40%). Every page is reviewed by a human. Not automated: publishing, deleting or merging pages, bulk title changes, and causal claims about refreshes.
Paper: section 6. Notebook: w07_action_playbook.ipynb.

In [5]:
print(pd.DataFrame(R["reasons"]))
print(pd.DataFrame(R["top10"]))

                          code                             action  pages
0         low_ctr_visible_page     rewrite title/meta and snippet   6216
1  low_engagement_visible_page  improve scanability and structure   1690
2           stale_visible_page                    refresh content   6575
3            thin_visible_page      expand with missing subtopics     52
   rank           reason_code                          action confidence  \
0     1  low_ctr_visible_page  rewrite title/meta and snippet       high   
1     2  low_ctr_visible_page  rewrite title/meta and snippet     medium   
2     3    stale_visible_page                 refresh content       high   
3     4  low_ctr_visible_page  rewrite title/meta and snippet     medium   
4     5  low_ctr_visible_page  rewrite title/meta and snippet     medium   
5     6    stale_visible_page                 refresh content       high   
6     7    stale_visible_page                 refresh content       high   
7     8    stale_visible_pa

## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

In [6]:
paths = ["docs/index.html", "docs/results.json", "submission/paper_url.txt",
         "work/notebooks/w04_baseline_score.ipynb", "work/notebooks/w05_model.ipynb",
         "work/notebooks/w06_validation_audit.ipynb", "work/notebooks/w07_action_playbook.ipynb"]
for p in paths:
    print("OK     " if os.path.exists(p) else "MISSING", p)
print("\npaper URL:", open("submission/paper_url.txt").read().strip()
      if os.path.exists("submission/paper_url.txt") else "(not set yet)")

OK      docs/index.html
OK      docs/results.json
OK      submission/paper_url.txt
OK      work/notebooks/w04_baseline_score.ipynb
OK      work/notebooks/w05_model.ipynb
OK      work/notebooks/w06_validation_audit.ipynb
OK      work/notebooks/w07_action_playbook.ipynb

paper URL: https://ayustar.github.io/flyrank-ml-internship-starter/


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [ ] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [ ] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.
